# Business Empire — Final Pipeline Notebook

**Author:** Maiky Artazos  
**Course:** RCEL 506 — Data Science Final  
**Date:** 2026-04-28

## Business problem

Manual game balancing does not scale. A team designing a 93-card strategy game cannot manually playtest every interaction between cards, strategies, draw orders, board positions, and bot opponents. The combinatorial test surface is too large to cover before the next physical playtest.

**Project goal:** use card data, seeded simulations, and interpretable models to find balance risks before physical playtesting. Deck metrics are loaded from the report files; the notebook also includes a smaller recompute to show the method runs end to end.

## What this notebook does

1. Loads real card data and shows EDA  
2. Runs a seeded Monte Carlo tournament to produce `win_bias` labels  
3. Loads the reported **naive baseline vs ridge** metrics used in the deck  
4. Recomputes a smaller ridge model with **out-of-fold cross-validation**  
5. Interprets coefficients through the lens of business meaning  
6. Compares **baseline vs final** on board balance and bot ML benchmarks  
7. Documents limitations

Total runtime: under 2 minutes on a fresh kernel.

In [ ]:
# Setup: imports, paths, seed.
from __future__ import annotations
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT.parent / "src").exists():
    SRC_DIR = ROOT.parent / "src"
    REPORTS_DIR = ROOT.parent / "reports"
    BOT_REPORTS_DIR = ROOT.parent / "bot-ml" / "reports"
    BOT_ARTIFACTS_DIR = ROOT.parent / "bot-ml" / "artifacts"
elif (ROOT / "src").exists():
    SRC_DIR = ROOT / "src"
    REPORTS_DIR = ROOT / "reports"
    BOT_REPORTS_DIR = ROOT / "bot-ml" / "reports"
    BOT_ARTIFACTS_DIR = ROOT / "bot-ml" / "artifacts"
else:
    SRC_DIR = ROOT
    REPORTS_DIR = ROOT / "optimizer_outputs" / "unified"
    BOT_REPORTS_DIR = Path.home() / "Documents" / "business-empire" / "training" / "bot-ml" / "reports"
    BOT_ARTIFACTS_DIR = Path.home() / "Documents" / "business-empire" / "training" / "bot-ml" / "artifacts"
sys.path.insert(0, str(SRC_DIR))

SEED = 20260414
np.random.seed(SEED)
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)
print(f"src directory: {SRC_DIR}")
REPORT_SUMMARY_PATH = REPORTS_DIR / "balance_summary.json"
print(f"reports directory: {REPORTS_DIR}")
print(f"report summary: {REPORT_SUMMARY_PATH}")

## 1. Load real card data

We parse the 93 card markdown files used by the playable app. The card pipeline is intentionally separated from the model so the dataset is reviewable independent of the ML.

In [ ]:
from card_parser import parse_all_cards, compute_derived_features

cards_df = compute_derived_features(parse_all_cards())
print(f"shape: {cards_df.shape}")
print("\ncards by type:")
print(cards_df["type"].value_counts())
print("\nbusiness cards by industry:")
business_df = cards_df[cards_df["type"].str.title() == "Business"]
print(business_df["industry"].value_counts())
print(f"\nphysical vs digital: {business_df['mode'].value_counts().to_dict()}")

## 2. Simulation metrics

We run a small seeded strategy tournament. Each card's `win_bias` is the simulator's view of how often the card appears in winning decks compared to a uniform expectation. This is the label the surrogate model will try to predict.

The simulator provides the target metrics. The model below is a diagnostic tool, not an acceptance gate.

In [ ]:
from game_engine_v3 import GameEngine, GameConfig, evaluate_strategy_tournament

engine = GameEngine(cards_df, GameConfig(board_enabled=True))
metrics = evaluate_strategy_tournament(
    engine=engine,
    strategies=["Random", "Greedy_VP", "Cash_Machine", "Scale_Rush"],
    n_simulations=24,
    base_seed=SEED,
    include_card_usage=True,
    include_matchup_results=False,
    verbose=False,
)
card_usage = metrics["card_usage"]
print(f"strategies played: {sorted(metrics['strategy_stats']['strategy'].unique())}")
print(f"card_usage rows: {len(card_usage)}  (cards observed in tournament)")
card_usage[["card_id", "usage_rate", "win_deck_rate", "loss_deck_rate", "win_bias"]].head(8)

## 3. Reported final metrics, then a fast recompute

The deck, README, and final report use the metrics from `reports/balance_summary.json`. The recompute below uses a smaller tournament so the method runs quickly. If the recompute values differ slightly, the report remains the deck metric source.


In [ ]:
# Load the model-comparison numbers used in the deck and README.
if not REPORT_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Canonical summary not found: {REPORT_SUMMARY_PATH}")
report_summary = json.loads(REPORT_SUMMARY_PATH.read_text(encoding="utf-8"))
report_ml = report_summary["ml_surrogate"]
report_table = pd.DataFrame([
    report_ml["baseline_metrics"],
    report_ml["oof_metrics"],
], index=["reported_naive_mean", "reported_ridge_oof"])[["mae", "rmse", "r2", "correlation"]]
print("Canonical final metrics used in the deck:")
print(report_table.round(4))
report_lift = (report_ml["baseline_metrics"]["mae"] - report_ml["oof_metrics"]["mae"]) / report_ml["baseline_metrics"]["mae"]
print(f"\nReported MAE improvement over naive: {report_lift:.1%}")

# Fast recompute: same modeling method, smaller tournament for speed.
training = business_df.merge(
    card_usage[["card_id", "win_bias"]].rename(columns={"card_id": "id"}),
    on="id",
    how="left",
).copy()
training["win_bias"] = pd.to_numeric(training["win_bias"], errors="coerce").fillna(0.0)
y = training["win_bias"].to_numpy(dtype=float)

naive_pred = np.full_like(y, fill_value=float(y.mean()))
naive_mae = float(np.mean(np.abs(y - naive_pred)))
print("\nFast recompute target summary:")
print(f"N business cards: {len(y)}")
print(f"target mean (win_bias): {y.mean():+.4f}")
print(f"target std:             {y.std():+.4f}")
print(f"fast recompute naive MAE: {naive_mae:.4f}")


## 4. Ridge regression — solved by hand

We implement ridge regression from first principles instead of calling `sklearn`. The closed-form solution is

$$\hat{w} = (X^\top X + \alpha I)^{-1} X^\top y$$

We standardize `X` (zero mean, unit variance per column) before the solve so coefficients are directly comparable.

**Model note:** with N=56 business cards and high simulation noise, the expected improvement is modest. The surrogate model flags suspicious cards; it does not replace simulation.

In [ ]:
NUMERIC_FEATURES = (
    "cost", "income", "valuation_points", "upkeep", "exit_value",
    "staff_min", "staff_opt", "income_scaled", "income_opt",
    "total_launch_cost", "avg_income", "max_income",
    "effective_roi", "payback_breaks",
    "synergy_output", "synergy_input", "synergy_links",
    "complexity_score", "tag_count", "requirement_count", "synergy_count",
    "time_delay", "effort", "likelihood",
)
CATEGORICAL_FEATURES = ("industry", "tier", "mode", "tempo")

def build_design_matrix(df: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    numeric_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
    numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    categorical_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]
    categorical = pd.get_dummies(
        df[categorical_cols].fillna("Unknown").astype(str),
        prefix=categorical_cols,
        dtype=float,
    )
    features = pd.concat([numeric, categorical], axis=1).astype(float)
    keep = features.columns[features.nunique(dropna=False) > 1]
    features = features[keep]
    return features.to_numpy(dtype=float), list(features.columns)

def fit_ridge(X: np.ndarray, y: np.ndarray, alpha: float = 1.5):
    """Standardized closed-form ridge. Returns intercept, weights (in original units),
    standardized weights, and the feature mean/scale used for prediction."""
    feat_mean = X.mean(axis=0)
    feat_scale = X.std(axis=0, ddof=0)
    safe_scale = np.where(feat_scale <= 1e-12, 1.0, feat_scale)
    Xs = (X - feat_mean) / safe_scale
    y_mean = float(y.mean())
    yc = y - y_mean
    A = Xs.T @ Xs + alpha * np.eye(Xs.shape[1])
    w_std = np.linalg.solve(A, Xs.T @ yc)
    w = w_std / safe_scale
    intercept = y_mean - float(feat_mean @ w)
    return intercept, w, w_std, feat_mean, safe_scale

def predict(X: np.ndarray, intercept: float, w: np.ndarray) -> np.ndarray:
    return intercept + (X @ w)

X, feature_names = build_design_matrix(training)
intercept, w, w_std, feat_mean, feat_scale = fit_ridge(X, y, alpha=1.5)
in_sample_pred = predict(X, intercept, w)
in_sample_mae = float(np.mean(np.abs(y - in_sample_pred)))
print(f"design matrix: {X.shape[0]} rows x {X.shape[1]} features")
print(f"in-sample MAE (full fit): {in_sample_mae:.4f}")
print("in-sample MAE is optimistic; the OOF estimate below is the better comparison.")

## 5. Fast recompute: out-of-fold validation

This cell reruns the ridge method at a smaller scale. These values are not the deck numbers unless they match the report; the deck numbers are loaded above from `reports/balance_summary.json`.


In [ ]:
def make_folds(n_rows: int, n_folds: int, seed: int) -> list[np.ndarray]:
    indices = np.arange(n_rows)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    return [fold.astype(int) for fold in np.array_split(indices, n_folds) if len(fold) > 0]

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    err = y_true - y_pred
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    total_var = float(np.sum((y_true - y_true.mean()) ** 2))
    residual_var = float(np.sum(err ** 2))
    r2 = 0.0 if total_var <= 1e-12 else 1.0 - residual_var / total_var
    if np.std(y_true) <= 1e-12 or np.std(y_pred) <= 1e-12:
        corr = 0.0
    else:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])
    return {"mae": round(mae, 4), "rmse": round(rmse, 4), "r2": round(r2, 4), "correlation": round(corr, 4)}

folds = make_folds(len(y), n_folds=5, seed=SEED)
oof = np.zeros_like(y)
for val_idx in folds:
    train_mask = np.ones(len(y), dtype=bool)
    train_mask[val_idx] = False
    fold_intercept, fold_w, _, _, _ = fit_ridge(X[train_mask], y[train_mask], alpha=1.5)
    oof[val_idx] = predict(X[val_idx], fold_intercept, fold_w)

naive_metrics = regression_metrics(y, naive_pred)
ridge_metrics = regression_metrics(y, oof)
comparison = pd.DataFrame([naive_metrics, ridge_metrics], index=["naive_mean", "ridge_oof"])
print("Fast recompute only - final report metrics are loaded above:\n")
print(comparison)
lift = (naive_metrics["mae"] - ridge_metrics["mae"]) / naive_metrics["mae"]
print(f"\nFast recompute MAE improvement over naive: {lift:.1%}")
print("Canonical final claim remains: MAE 0.0180 -> 0.0163, R-squared 0.0905.")

## 6. Coefficient interpretation through business meaning

**Coefficient interpretation.** This section connects feature direction to operational meaning.

Each row below answers exactly that question for the top features. Standardized coefficients are directly comparable: a unit-magnitude standardized coefficient means a 1-σ increase in the feature shifts predicted `win_bias` by that amount.

In [ ]:
BUSINESS_MEANING = {
    "effective_roi": "Income per launch cost. Higher ROI cards win more — the bot economy rewards efficiency.",
    "payback_breaks": "Turns to recover launch cost. Surprisingly positive — cards that take longer to pay back tend to be late-game scaling plays that win when they land.",
    "likelihood": "Card's printed reliability. Slightly negative coefficient suggests safe cards under-perform high-variance scaling cards in the metagame.",
    "income_scaled": "Income at staff_opt. Weak negative — overpaying for staffing is not always the winning move.",
    "effort": "Required action effort. Mild negative — high-effort cards crowd out other plays.",
    "valuation_points": "End-game VP. The bot weights VP and income as complements, not substitutes.",
    "income": "Per-break income. Closely correlated with effective_roi.",
    "cost": "Launch cost. Penalized via interaction with ROI.",
    "complexity_score": "Rule-load proxy. Captures whether the card is hard to play correctly.",
}

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": w,
    "std_coef": w_std,
})
coef_df["abs_std"] = coef_df["std_coef"].abs()
coef_df = coef_df.sort_values("abs_std", ascending=False).reset_index(drop=True)
coef_df["business_meaning"] = coef_df["feature"].map(BUSINESS_MEANING).fillna("(category dummy / supporting feature)")

top = coef_df.head(8)[["feature", "std_coef", "business_meaning"]].copy()
top["std_coef"] = top["std_coef"].round(4)
print("Top 8 features by absolute standardized coefficient (surrogate model on win_bias):\n")
for _, row in top.iterrows():
    sign = "+" if row["std_coef"] >= 0 else "-"
    print(f"  {row['feature']:<22} {sign}{abs(row['std_coef']):.4f}   {row['business_meaning']}")

### Operational interpretation

The coefficient table explains direction and trade-off; the performance claim remains the report result: naive MAE `0.0180`, ridge OOF MAE `0.0163`, and OOF R-squared `0.0905`.

**Why this is still useful:**

- N=56 business cards is genuinely small.  
- `win_bias` has simulation variance because each card appears in only a fraction of seeded games.  
- A modest R-squared is expected when noise dominates signal but the signal direction is useful.  
- The value is in the baseline comparison and coefficient interpretation, not in claiming a large R-squared.


## 7. Bot ML scorer — coefficients with direct game meaning

The Expo app uses a separate, smaller ML model: a 10-feature standardized linear scorer that ranks legal candidate **actions** (not cards). Its coefficients are useful because they map directly to business decisions a player can recognize.

The notebook reads them from the artifact JSON below. If the artifact is not present, it falls back to the values committed with this repository.

In [ ]:
BOT_FALLBACK = [
    ("actionIncome", 0.88, "Bot prefers revenue-generating actions."),
    ("actionVP", 0.83, "Victory points weighted near-equal to income."),
    ("actionCost", -0.63, "Cost is correctly penalized at ~70% of income."),
    ("placementAffinity", 0.25, "Physical board fit is tactical, not strategic."),
    ("brand", 0.16, "Bot is mildly more aggressive when brand is healthy."),
]

model_path = BOT_ARTIFACTS_DIR / "optimized-expert-model.json"
if model_path.exists():
    payload = json.loads(model_path.read_text(encoding="utf-8"))
    coefs = payload.get("coefficients", {})
    meaning = {f: m for f, _, m in BOT_FALLBACK}
    bot_rows = sorted(coefs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
    bot_table = [(name, float(value), meaning.get(name, "Standardized linear effect.")) for name, value in bot_rows]
    source = "loaded from optimized-expert-model.json"
else:
    bot_table = BOT_FALLBACK
    source = "fallback (artifact not found at expected path)"

print(f"Source: {source}\n")
fig, ax = plt.subplots(figsize=(8.5, 3.5))
labels = [row[0] for row in bot_table][::-1]
values = [row[1] for row in bot_table][::-1]
colors = ["#2E5F4D" if v >= 0 else "#A94532" for v in values]
ax.barh(labels, values, color=colors, edgecolor="#1D2824")
ax.axvline(0, color="#1D2824", linewidth=0.8)
ax.set_xlabel("standardized coefficient")
ax.set_title("Bot ML scorer: how the model reads a decision")
for label, val in zip(labels, values):
    ax.text(val + (0.02 if val >= 0 else -0.02), label, f"{val:+.2f}",
            va="center", ha="left" if val >= 0 else "right", fontsize=10, fontweight="bold")
ax.set_xlim(-1.05, 1.05)
plt.tight_layout()
plt.show()
print("\nBusiness translation:")
for name, value, meaning_text in bot_table:
    sign = "+" if value >= 0 else "-"
    print(f"  {name:<20} {sign}{abs(value):.2f}   {meaning_text}")

## 8. Board balance — baseline vs final (the clearest improvement)

The pipeline produces a separate report on spatial board balance. The baseline uses the original board configuration; the final reflects the accepted optimization (one feature weight changed: `plaza_adjacent` from 0.35 to 0.30).

This is the strongest "baseline vs final" story in the project: zone gap reduced ~52%.

In [ ]:
summary_path = REPORTS_DIR / "balance_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    board = summary["board"]
    rows = [
        ("overall balance (higher better)",
         board["baseline"]["mean_overall_balance"],
         board["final_verification"]["mean_overall_balance"]),
        ("zone gap (lower better)",
         board["candidate_dynamic_before"]["zone_gap"],
         board["candidate_dynamic_after"]["zone_gap"]),
        ("weakest-zone win rate (higher better)",
         board["candidate_dynamic_before"]["weakest_zone_win_rate"],
         board["candidate_dynamic_after"]["weakest_zone_win_rate"]),
    ]
    board_df = pd.DataFrame(rows, columns=["metric", "baseline", "final"])
    board_df["delta"] = board_df["final"] - board_df["baseline"]
    print("Board balance (baseline vs final):\n")
    print(board_df.round(4).to_string(index=False))
else:
    print(f"balance summary not found at {summary_path}")
    print("run src/run_balance_pipeline.py to regenerate")

## 9. Bot ML benchmark — seat-rotated, 400 games per policy

The bot benchmark is run inside the playable Expo app and reported here as a snapshot. Every policy plays from every seat on the same seeds so seat-order artifacts cannot bias the result.

**Honest finding:** the **hard tuned heuristic outperforms the expert ML scorer** (+8.51 vs +2.37 score lift over the medium baseline). Expert ML still beats the baseline; the heuristic is currently stronger. On small-sample action ranking with well-engineered features, search beats parametric models — that's a real data-science finding, not a hedge.

In [ ]:
bot_summary_path = BOT_REPORTS_DIR / "bot-ml-summary.md"
if bot_summary_path.exists():
    rows = []
    headers: list[str] = []
    for line in bot_summary_path.read_text(encoding="utf-8").splitlines():
        if not line.startswith("|"):
            continue
        cells = [c.strip() for c in line.strip("|").split("|")]
        if not cells or set(cells[0]) == {"-"}:
            continue
        if cells[0] == "Policy":
            headers = cells
            continue
        if headers and len(cells) == len(headers):
            rows.append(cells)
    bot_df = pd.DataFrame(rows, columns=headers)
    print("Seat-rotated bot benchmark (lower is better for invalid actions; medium = baseline):\n")
    print(bot_df[["Policy", "Win rate", "Avg score", "Score diff vs same-seat medium", "Invalid actions"]].to_string(index=False))
else:
    print(f"bot summary not found at {bot_summary_path}")
    print("see bot-ml/reports/ in this repository")

## 10. Limitations and how to extend

**Honest limitations**

- N=56 business cards is small. R² of ~0.09 on the surrogate is expected, not concerning.
- The card balance optimizer accepted **0** edits after confirmation gates. This is intentional conservatism — the search trace at `reports/search_trace.csv` shows 145 candidates evaluated and rejected. We refuse to overfit short-run simulations.
- The expert ML bot does **not** beat the tuned heuristic. The deck and bot report show that result directly.

**Operational trade-offs**

- Increasing tournament `n_simulations` reduces `win_bias` variance but increases runtime linearly.
- Decreasing `alpha` (ridge regularization) lets the model fit harder but risks overfitting on N=56.
- The bot scorer uses 10 features by design — adding more would require re-deriving means/scales and re-validating against the medium baseline.

**How to extend**

- Replace the linear scorer with a small gradient-boosted model when N grows past ~200.
- Expand the strategy pool beyond 4 named strategies for a richer metagame label.
- Add a confidence interval to the OOF MAE via repeated K-fold with different seeds.

## Repository map

```
Final-Presentation-RCEL-506/
├── README.md                          # executive summary + cross-reference
├── slides/                            # final deck and card contact sheet
├── notebooks/final_pipeline.ipynb     # this file
├── src/                               # supporting Python modules + tests
├── data/cards_dataset.csv             # the dataset
├── app/                               # Streamlit app
├── bot-ml/                            # frozen bot ML snapshot from the app
└── reports/                           # generated artifacts (balance, OWLET)
```

**Maiky Artazos — Spring 2026**